# 02.HuggingFace远程推理嵌入模型

知识点讲解：HuggingFaceEndpointEmbeddings 远程推理嵌入模型

1. 本地加载 vs 远程推理
   - HuggingFaceEmbeddings（上一个示例）：把模型权重下载到本机，用本地算力推理
       优点：完全离线、无调用次数限制、数据不外发
       缺点：占用磁盘与内存、依赖本机算力、首次需下载大文件
   - HuggingFaceEndpointEmbeddings（本示例）：只发 HTTP 请求，推理在远端完成
       优点：零本地算力消耗、无需下载权重、启动快、客户端依赖轻
       缺点：依赖网络、受限流约束、数据需外发、免费额度有限
   - 部分模型的文件比较大，如果只是短期内调试，可以考虑使用 HuggingFace
     提供的远程嵌入模型

2. HuggingFace Inference API 两种形态
   - Serverless Inference API（免费/共享）：
     地址形如 https://api-inference.huggingface.co/pipeline/feature-extraction/{model}
     由 HuggingFace 托管公共模型，有速率限制；冷启动时首次请求可能返回 503，
     表示模型正在加载（loading），需要重试
   - Dedicated Inference Endpoint（付费/独占）：
     用户自行部署的专属推理端点，有独立 URL 与稳定性能，
     通过 model="https://xxx.endpoints.huggingface.cloud" 直接传入完整地址

3. 鉴权配置
   - 需要 HuggingFace Access Token，在 .env 中配置环境变量：
     HUGGINGFACEHUB_API_TOKEN=hf_xxxxxxxxxxxx
   - Token 在 https://huggingface.co/settings/tokens 创建，读权限（read）即可
   - 也可通过构造参数 huggingfacehub_api_token="hf_xxx" 显式传入
   - 需要在 Hugging Face 官网（https://huggingface.co/）的 setting 中
     添加对应的访问秘钥，并配置到 .env 文件中

4. 重要参数说明
   - model     : 模型标识（组织名/模型名）或完整的 Inference Endpoint URL
   - task      : 推理任务类型，嵌入场景默认为 "feature-extraction"
   - huggingfacehub_api_token : 访问令牌，不传则读取环境变量
   - model_kwargs : 附加到请求体的参数

5. 与本地模型的一致性
   - 同一个模型（如 all-MiniLM-L12-v2）无论本地推理还是远程推理，
     输出维度相同（384），语义空间也相同，理论上向量可以互换使用
   - 因此可以在开发期用远程 API 快速验证，上线后切换为本地部署，
     无需重建向量数据库（前提是模型版本完全一致）

6. 最佳实践建议
   - 远程推理存在网络抖动风险，生产环境务必配合 with_retry() 或重试装饰器
   - 免费 Serverless API 有严格限流，不适合批量嵌入大规模知识库，
     大批量任务建议改用本地模型或自建专属端点
   - 结合 CacheBackedEmbeddings 缓存结果，可显著减少重复请求
   - 敏感数据不要走公共 Serverless API，应使用本地模型或私有化端点
   - 在本地服务器上就无需配置对应的文本嵌入模型了


In [ ]:
import dotenv
from langchain_huggingface import HuggingFaceEndpointEmbeddings
dotenv.load_dotenv()
embeddings = HuggingFaceEndpointEmbeddings(model="sentence-transformers/all-MiniLM-L12-v2")
query_vector = embeddings.embed_query("你好，我是慕小课，我喜欢打篮球游泳")
print(query_vector)
print(len(query_vector))